## B05 阶段：建立项目基线（追加到原项目）

保留阶段一草稿，将本片段追加到原 project.ipynb，不覆盖原文件。这里只读取 120 行开发数据；保留测试不读取。多数类基线不使用特征值，因此暂不需要填补；后续候选模型仍须按训练范围处理缺失。

运行前沿用原计划填模式、种子和主指标。不比较两种划分后挑分数；计划尚未明确时先记录理由再运行。

In [ ]:
MODE = "cv"  # 原计划为固定验证时改为 'holdout'，只运行选定模式
SEED = 42  # 沿用原计划
PRIMARY_METRIC = "accuracy"  # 可为accuracy/recall/precision；必须先说明任务理由
assert MODE in ["cv", "holdout"]
assert PRIMARY_METRIC in ["accuracy", "recall", "precision"]

**与阶段一计划一致的依据／必要变更及事前理由：**（运行前填写）

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend(
        [
            parent / "development.csv",
            parent / "project/development.csv",
            parent / "units/decision-tree/project/development.csv",
            parent / "units/boosting/project/development.csv",
        ]
    )
project_path = next((p for p in candidates if p.is_file()), None)
if project_path is None:
    raise FileNotFoundError("把片段追加到原project.ipynb，并保留development.csv")
development = pd.read_csv(project_path)
target = development["defective"].to_numpy(dtype=int)
assert (
    len(development) == 120 and development.batch_id.is_unique and set(target) == {0, 1}
)
indices = np.arange(len(target))
dummy_X = np.zeros((len(target), 1))
if MODE == "cv":
    splits = list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED).split(
            dummy_X, target
        )
    )
else:
    ti, vi = train_test_split(
        indices, test_size=0.25, random_state=SEED, stratify=target
    )
    splits = [(ti, vi)]
baseline_protocol = {
    "mode": MODE,
    "seed": int(SEED),
    "primary_metric": PRIMARY_METRIC,
    "split_indices": [(ti.tolist(), vi.tolist()) for ti, vi in splits],
}
baseline_predictions = np.full(len(target), -1, dtype=int)
coverage = np.zeros(len(target), dtype=int)
stage_rows = []
for i, (ti, vi) in enumerate(splits, 1):
    assert set(ti).isdisjoint(vi)
    baseline = DummyClassifier(strategy="most_frequent").fit(dummy_X[ti], target[ti])
    pred = baseline.predict(dummy_X[vi])
    baseline_predictions[vi] = pred
    coverage[vi] += 1
    tn, fp, fn, tp = confusion_matrix(target[vi], pred, labels=[0, 1]).ravel()
    stage_rows.append(
        {
            "part": i,
            "train_rows": len(ti),
            "validation_rows": len(vi),
            "accuracy": (tp + tn) / len(vi),
            "recall": tp / (tp + fn) if tp + fn else np.nan,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
        }
    )
baseline_results = pd.DataFrame(stage_rows)
display(baseline_results)
assert coverage.max() == 1
evaluated = coverage == 1
assert evaluated.sum() == (120 if MODE == "cv" else 30)
tn, fp, fn, tp = confusion_matrix(
    target[evaluated], baseline_predictions[evaluated], labels=[0, 1]
).ravel()
baseline_summary = {
    "accuracy": (tp + tn) / evaluated.sum(),
    "recall": tp / (tp + fn) if tp + fn else np.nan,
    "precision": tp / (tp + fp) if tp + fp else np.nan,
}
print(
    "模式：",
    MODE,
    "主指标：",
    PRIMARY_METRIC,
    "开发内汇总值：",
    baseline_summary[PRIMARY_METRIC],
)
print("NaN表示指标分母为0、未定义；不要当作有效高分或改挑有利指标。")
print("未读取或评价任何保留测试资料。")

**阶段解释：**

1. 引用主指标与混淆计数说明基线；若只预测正常而漏掉全部不合格，解释其局限。
2. 填写拟合范围、划分模式和随机种子，说明这只是开发内评价。
3. 下一阶段计划比较哪一个有依据的候选？本次只记录，不额外训练。

继续使用同一个项目 Notebook，保存并备份。